# Pré-treino australiano — cinco minutos

Resolução real de **cinco minutos**, horizonte **t+5 minutos**. Os nomes dos arquivos foram mantidos por solicitação do usuário; o conteúdo foi migrado de diário para intradiário.

O alvo e a persistência (t−5 min) precisam ser originalmente medidos. Cada etapa de preenchimento causal usa até 15 minutos; no Brasil a combinação proxy + ffill pode atingir 30 minutos somente no histórico: nunca se interpola usando ponto posterior. Não se concatenam blocos interrompidos. A avaliação principal é diurna (elevação solar > 0); a avaliação 24h é complementar. Zero noturno não deve inflar a precisão.

As entradas principais são potência AC em p.u., hora local e dia do ano em seno/cosseno. O vento continua excluído. ERA5 nativa horária é auxiliar e não entra nos modelos principais. Estatísticas de normalização vêm apenas do treino australiano; alvos de validação/teste não ajustam escalas.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Image,Markdown
RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/energia_transfer').is_dir())
if str(RAIZ/'src') not in sys.path: sys.path.insert(0,str(RAIZ/'src'))
PASTA=RAIZ/'experimentos/transfer_learning_australia_brasil'
AU=RAIZ/'experimentos/australia_deep_learning'
REPORT=RAIZ/'reports/intradiario'
from energia_transfer.transfer_learning.dados import carregar,particao
_,CONFIG,CONTRATO=carregar(PASTA)
def exibir(t): display(t.round(5))


In [2]:
metricas=pd.read_csv(PASTA/'resultados/pretreino_metricas.csv')
exibir(metricas.groupby(['arquitetura','lag','particao']).agg(RMSE_pu=('RMSE_pu','mean'),DP=('RMSE_pu','std'),MAE_pu=('MAE_pu','mean'),n=('n','first')))
exibir(pd.read_csv(PASTA/'resultados/pretreino_execucoes.csv').head(15))


RMSE_pu       DP   MAE_pu      n
arquitetura  lag particao                                   
lstm_direta  6   treino     0.05696  0.00028  0.02749  40747
                 validacao  0.03973  0.00024  0.01790  46840
             12  treino     0.05657  0.00051  0.02744  40704
                 validacao  0.03948  0.00028  0.01793  46840
mlp_direta   6   treino     0.05548  0.00040  0.02649  40747
                 validacao  0.03970  0.00021  0.01822  46840
             12  treino     0.05520  0.00044  0.02712  40704
                 validacao  0.03960  0.00037  0.01853  46840
mlp_residual 6   treino     0.05499  0.00016  0.02568  40747
                 validacao  0.03898  0.00005  0.01681  46840
             12  treino     0.05459  0.00019  0.02579  40704
                 validacao  0.03882  0.00008  0.01708  46840

,arquitetura,lag,seed,melhor_epoca,epocas,parametros
0,mlp_direta,6,42,23,25,1537
1,mlp_direta,6,7,13,18,1537
2,mlp_direta,6,21,24,25,1537
3,mlp_direta,6,84,22,25,1537
4,mlp_direta,6,123,24,25,1537
5,mlp_residual,6,42,10,15,1537
6,mlp_residual,6,7,13,18,1537
7,mlp_residual,6,21,22,25,1537
8,mlp_residual,6,84,19,24,1537
9,mlp_residual,6,123,17,22,1537


## Reexecutar cálculos

Use o terminal para evitar treinos longos ocultos em uma célula:

```powershell
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py preparar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py pretreinar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py adaptar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py avaliar
```

Históricos de 30 e 60 minutos (6/12 passos). São cinco sementes, MLP direta, MLP residual e LSTM direta. Treino fonte: uma de cada 12 janelas diurnas válidas, por custo de CPU, mantendo 2013–2022; validação 2023 e teste 2024–2025 em todos os instantes elegíveis. Treino brasileiro até setembro, validação outubro, teste novembro–dezembro de 2025. Os dados são autocorrelacionados: número de instantes não equivale a observações independentes. A configuração fixa máximos de 25 épocas fonte e 40 na adaptação; são experimentos iniciais, sem busca no teste.
